In [1]:
import os
import pandas as pd
import numpy as np
from socceraction.atomic.vaep.formula import value

In [ ]:
features_dfs = []

for features_file in os.listdir("../data/features"):
    features_dfs.append(pd.read_parquet(f"../data/features/{features_file}"))

labels_dfs = []

for labels_file in os.listdir("../data/labels"):
    labels_dfs.append(pd.read_parquet(f"../data/labels/{labels_file}"))

actions_dfs = []

for actions_file in os.listdir("../data/actions"):
    actions_dfs.append(pd.read_parquet(f"../data/actions/{actions_file}", columns=["game_id", "action_id", "team_id", "type_id", "player_id"]))

events_dfs = []

for events_file in os.listdir("../data/events"):
    events_dfs.append(pd.read_parquet(f"../data/events/{events_file}", columns=["player_id", "player", "minutes"]))

events_df = pd.concat(events_dfs)
events_df = events_df.drop_duplicates().dropna()
actions_df = pd.concat(actions_dfs)
X = pd.concat(features_dfs)
y = pd.concat(labels_dfs)

In [ ]:
X = pd.merge(
    X,
    actions_df,
    on=["game_id", "action_id"],
    how="left"
)

X = pd.merge(
    X,
    events_df,
    on="player_id",
    how="left"
)

In [ ]:
from socceraction.atomic.spadl import config as atomicspadl

X["type_name"] = X["type_id"].apply(lambda i: atomicspadl.actiontypes[i])

In [ ]:
for col in X.select_dtypes(include="bool").columns:
    X[col] = X[col].astype("int8")

max_pitch_distance = np.sqrt(120**2 + 80**2)

for col in [
    "actor_minimum_distance_to_opponent",
    "actor_minimum_distance_to_teammate",
    "actor_mean_distance_to_opponent",
    "actor_mean_distance_to_teammate",
]:
    X[col] = X[col].fillna(max_pitch_distance)

for col in [
    "actor_stdev_distance_to_opponent",
    "actor_stdev_distance_to_teammate",
    "opponent_convex_hull_area",
    "teammate_convex_hull_area"
]:
    X[col] = X[col].fillna(0)

In [ ]:
from joblib import load

xgboost_all_concedes = load("../models/classifiers/xgboost_all_concedes.pkl")
xgboost_all_scores = load("../models/classifiers/xgboost_all_scores.pkl")

In [ ]:
exclude_cols = ["game_id", "action_id", "team_id", "type_id", "type_name", "player_id", "player"]

In [ ]:
xgboost_all_concedes.predict_proba(X.drop(columns=exclude_cols))[:, 1]

In [ ]:
X_prediction = X.copy()

X_prediction["concede_prob"] = xgboost_all_concedes.predict_proba(X.drop(columns=exclude_cols))[:, 1]
X_prediction["score_prob"] = xgboost_all_scores.predict_proba(X.drop(columns=exclude_cols))[:, 1]

In [ ]:
X_prediction[["offensive_value", "defensive_value", "vaep_value"]] = value(
    X_prediction,
    X_prediction["score_prob"],
    X_prediction["concede_prob"]
)

In [ ]:
X_prediction[["player", "score_prob", "concede_prob", "offensive_value", "defensive_value", "vaep_value"]].groupby("player").mean().reset_index()